In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink
import math
from itertools import product
import matplotlib.pyplot as plt
import numpy as np
import csv
import pandas as pd
import scipy.stats as stats


def router(
    b_0_list=[None],
    b_1_list=[None],
    w0_list=[None],
    m_list=[None],
    A_list=[None],
    B_list=[None]
):

    
    
    if b_0_list != [None] and b_1_list != [None]:
        return crush_AB(
            b_0_list=b_0_list,
            b_1_list=b_1_list,
            w0_list=w0_list,
            m_list=m_list
        )

    if A_list != [None] and B_list != [None]:
        return crush_bob1(
            A_list=A_list,
            B_list=B_list,
            w0_list=w0_list,
            m_list=m_list
        )

    return pd.DataFrame()


def crush_AB(
    b_0_list=[None],
    b_1_list=[None],
    w0_list=[None],
    m_list=[None],  
 
):
    """
    Inputs:
      b0: speed intercept (mph, kmph, fps)
      b1: speed per unit crush ()
      w0:  crush zone width (in, ft, mm, in)
      m:  mass (slugs or kg)


     """
 


    results = []
    for b_0, b_1, w0, m in product(b_0_list, b_1_list, w0_list, m_list):
        
        if w0_list == [None] or m_list== [None]:
            continue
            
        if w0 == 0:
            continue
        A = (m / w0) * (b_0 * b_1)
        B = (m / w0) * (b_1 * b_1)

        if B == 0:
            continue

        G = (A*A)/(B)
        results.append({
            'b0': b_0,
            'b1': b_1,
            'width': w0,
            'mass': m,
            'A': A,
            'B': B,
            'G': G
        })

    return pd.DataFrame(results)



def crush_bob1(
    A_list=[None],
    B_list=[None],
    w0_list=[None],
    m_list=[None]
):

    results = []

    for A, B, w0, m in product(A_list, B_list, w0_list, m_list):
        
        if w0_list == [None] or m_list== [None]:
            continue

        if B == 0 or w0 == 0 or m == 0:
            continue

        b_1 = math.sqrt(B * (w0 / m))
        b_0 = A * (w0 / m) / b_1
        G = (A * A) / B

        results.append({
            'b0': b_0,
            'b1': b_1,
            'width': w0,
            'mass': m,
            'A': A,
            'B': B,
            'G': G
        })

    return pd.DataFrame(results)




def safe_castv3(val):
    if val is None or val.strip().lower() == 'none' or val.strip() == '':
        return [None]
    try:
        result = eval(val, {"np": np, "__builtins__": {}})
        if isinstance(result, (list, np.ndarray)):
            return list(result)
        elif isinstance(result, tuple):
            return list(result)
        else:
            return [result]
    except:
        return [None]
        
def normalize(val):
    if isinstance(val, str):
        return safe_castv3(val)
    else:
        # already a list or number
        return val


In [2]:
# Define unit options
slope_b1_units = ['mph/ft', 'mph/in', 'mph/cm', 'mph/mm','fps/ft','fps/in', 'fps/cm', 'fps/mm','mps/ft','mps/in', 'mps/cm', 'mps/mm',
                  'kmph/ft', 'kmph/in', 'kmph/cm', 'kmph/mm']
speed_b0_units = ['mps', 'kmph', 'mph', 'fps']
time_units=['seconds', 'minutes', 'hours']
width_units=['feet','inch','yard','mm','cm','meter']
mass_units=['slug','kg','lbs']

# ---- New: input mode options ----
entry_modes = ['Manual / Comma / Bracket', 'NumPy Range', 'NumPy Linspace']


# ---- Helpers to create a mode-controlled input UI for each variable ----
def make_mode_input(label, initial_text=''):
    mode_dd = widgets.Dropdown(options=entry_modes, value='Manual / Comma / Bracket',
                               description=f'{label} Mode:', style={'description_width': 'initial'})

    # Manual / comma / bracket text field
    manual_text = widgets.Text(value=initial_text, description=f'{label}:', style={'description_width': 'initial'})

    # NumPy Range widgets
    r_start = widgets.FloatText(value=0.0, description='Start:', style={'description_width': 'initial'})
    r_end   = widgets.FloatText(value=10.0, description='End:',   style={'description_width': 'initial'})
    r_step  = widgets.FloatText(value=1.0, description='Step:',  style={'description_width': 'initial'})

    # NumPy Linspace widgets
    l_start  = widgets.FloatText(value=0.0, description='Start:', style={'description_width': 'initial'})
    l_end    = widgets.FloatText(value=1.0, description='End:',   style={'description_width': 'initial'})
    l_points = widgets.IntText(value=5, description='Points:',    style={'description_width': 'initial'})

    # Container to swap the input controls based on mode
    input_container = widgets.VBox()

    def update_inputs(_=None):
        m = mode_dd.value
        if m == 'Manual / Comma / Bracket':
            input_container.children = [manual_text]
        elif m == 'NumPy Range':
            input_container.children = [widgets.HBox([r_start, r_end, r_step])]
        else:  # NumPy Linspace
            input_container.children = [widgets.HBox([l_start, l_end, l_points])]
    mode_dd.observe(update_inputs, names='value')
    update_inputs()

    # Getter that returns the parsed Python value (scalar or list)
    def getter():
        m = mode_dd.value
        try:
            if m == 'Manual / Comma / Bracket':
                txt = manual_text.value.strip()
                if txt == '':
                    return ''
                # Accept bracketed lists, single scalars, or comma lists without brackets
                # If plain comma list without brackets, wrap in brackets before safe_cast
                stripped = txt.replace(' ', '')
                if ',' in stripped and not (txt.startswith('[') and txt.endswith(']')):
                    return safe_castv3(f'[{txt}]')
                return safe_castv3(txt)
            elif m == 'NumPy Range':
                start = float(r_start.value)
                end   = float(r_end.value)
                step  = float(r_step.value)
                if step == 0:
                    return []
                return np.arange(start, end+step, step).tolist()
            else:  # NumPy Linspace
                start  = float(l_start.value)
                end    = float(l_end.value)
                points = int(l_points.value)
                if points <= 0:
                    return []
                return np.linspace(start, end, points).tolist()
        except Exception:
            return ''

    # Return the UI row pieces so we can place them next to unit dropdowns
    return mode_dd, input_container, getter, manual_text, (r_start, r_end, r_step), (l_start, l_end, l_points)





# ---- Build inputs (with modes) and units on the Calculate tab ----
# Distance
width_mode, width_input_box, get_width, _, _, _ = make_mode_input('Width', initial_text='[36,37]')
width_unit = widgets.Dropdown(options=width_units, value='inch', description='Width Unit:', style={'description_width': 'initial'})
width_row = widgets.HBox([width_input_box, width_unit, width_mode])

# Initial speed
b0_mode, b0_input_box, get_b0, _, _, _ = make_mode_input('b0', initial_text='5')
b0_unit = widgets.Dropdown(options=speed_b0_units, value='mph', description='b0 Speed Unit:', style={'description_width': 'initial'})
b0_row = widgets.HBox([b0_input_box, b0_unit, b0_mode])


# Mass
mass_mode, mass_input_box, get_mass, _, _, _ = make_mode_input('Mass', initial_text='2000')
mass_unit = widgets.Dropdown(options=mass_units, value='lbs', description='Mass Unit:', style={'description_width': 'initial'})
mass_row = widgets.HBox([ mass_input_box, mass_unit, mass_mode])

# Slope b1 units
b1_mode, b1_input_box, get_b1, _, _, _ = make_mode_input('b1', initial_text='5,5,6')
b1_unit = widgets.Dropdown(options=slope_b1_units, value='mph/in', description='b1 Slope Unit:', style={'description_width': 'initial'})
b1_row = widgets.HBox([ b1_input_box, b1_unit, b1_mode])

# A
A_mode, A_input_box, get_A, _, _, _ = make_mode_input('A', initial_text='')
A_unit = widgets.Text(value=f"{mass_unit.value}*{b0_unit.value}*{b1_unit.value}/{width_unit.value}",description="A Unit:")
A_row = widgets.HBox([ A_input_box, A_unit, A_mode])

# Slope b1 units
B_mode, B_input_box, get_B, _, _, _ = make_mode_input('B', initial_text='')
B_unit = widgets.Text(value=f"{mass_unit.value}*{b1_unit.value}²/{width_unit.value}",description="B Unit:")
B_row = widgets.HBox([ B_input_box, B_unit, B_mode])




# Output unit 
o_width_unit = widgets.Text(description="Width:", value=f"{width_unit.value}")
o_b0_unit = widgets.Text(description="b0 Speed:",  value=f"{b0_unit.value}")
o_mass_unit = widgets.Text(description="Mass:", value=f"{mass_unit.value}")
o_b1_unit = widgets.Text(description="b1 Slope:",  value=f"{b1_unit.value}")
o_A_unit = widgets.Text(value=f"{mass_unit.value}*{b0_unit.value}*{b1_unit.value}/{width_unit.value}",description="A Unit:")
o_B_unit = widgets.Text(value=f"{mass_unit.value}*{b1_unit.value}²/{width_unit.value}",description="B Unit:")

# Submit button and output
submit_btn = widgets.Button(description='Submit', button_style='success')
output = widgets.Output()

def on_submit(b):
    output.clear_output()
    with output:
        # Use mode-driven getters instead of raw text
        width_raw = get_width()
        b0_raw = get_b0()
        mass_raw = get_mass()
        b1_raw = get_b1()
        A_raw = get_A()
        B_raw = get_B()
 

        w0 = normalize(width_raw)
        b0 = normalize(b0_raw)
        m = normalize(mass_raw)
        b1 = normalize(b1_raw)
        A = normalize(A_raw)
        B = normalize(B_raw)


        if mass_unit.value == 'lbs':
            if isinstance(m, list):
               m = [x / 32.2 for x in m]
            else:
                m = m / 32.2
    

        print("\n Your Inputs \n")
        print("Width (w0):", w0)
        print("Mass (m):", m)
        print("b0:", b0)
        print("b1:", b1)
        print("A", A)
        print("B",B)
       
        
   

        # Compute
        df = router(b_0_list=b0,b_1_list=b1,w0_list=w0,m_list=m,A_list=A,B_list=B)

        
       
        
        print("\n Outputs \n")
        display(df)

        return w0, m, b0, b1, A, B, df

submit_btn.on_click(on_submit)

# Layout (Time-Dist. Calculate tab) — units on left, mode dropdown and input controls to the right
rows = [
    widgets.HBox([widgets.Label(value=" Must provide w0 and m; provide b0 and b1 or A and B.")]),
    width_row,
    mass_row,
    b0_row,
    b1_row,
    A_row,
    B_row
]

o_rows= [
    widgets.HBox([widgets.Label(value="List of Output Units")]),
    widgets.HBox([o_width_unit]),
    widgets.HBox([o_mass_unit]),
    widgets.HBox([o_b0_unit]),
    widgets.HBox([o_b1_unit]),
    widgets.HBox([o_A_unit]),
    widgets.HBox([o_B_unit])
]

html_text = """<pre style="white-space:pre-wrap">
Data Entry Guide (Now supports modes)

Modes (per field, right of unit dropdown):
 - Manual / Comma / Bracket: type a single number, a comma list i.e. 10,20,30, or a bracketed list i.e. [10,20,30]
 - NumPy Range: specify Start, End, Step (creates list via np.arange)
 - NumPy Linspace: specify Start, End, Points (creates list via np.linspace)


Importing Spreadsheets
This porgram accepts comma seaprated lists. Convert an excel column, row or any range to a comma separated list with excel command:
=TEXTJOIN(",",TRUE,A1:A5)
This would turn the column A from rows 1 to 5 into  a comma separated list you can paste in here
</pre>"""
def update_units(change=None):
    A_unit.value = (f"{mass_unit.value}*{b0_unit.value}*{b1_unit.value}/{width_unit.value}")
    B_unit.value = (f"{mass_unit.value}*{b1_unit.value}²/{width_unit.value}")
    o_width_unit.value = (f"{width_unit.value}")
    o_b0_unit.value = (f"{b0_unit.value}")
    o_mass_unit.value = (f"{mass_unit.value}")
    o_b1_unit.value = (f"{b1_unit.value}")
    o_A_unit.value =  (f"{mass_unit.value}*{b0_unit.value}*{b1_unit.value}/{width_unit.value}")
    o_B_unit.value = (f"{mass_unit.value}*{b1_unit.value}²/{width_unit.value}")

for w in [mass_unit, b0_unit, b1_unit, width_unit]:
    w.observe(update_units, names="value")
    
update_units()

Help = widgets.HTML(value=html_text)

o_units=widgets.VBox(o_rows)
Calculate = widgets.VBox(rows + [widgets.HBox([submit_btn]), o_units, output])

tab_nest = widgets.Tab()
tab_nest.children = [Calculate, Help]
tab_nest.set_title(0, 'Crush Coefficients')
tab_nest.set_title(1, 'Help')

display(tab_nest)


In [19]:
w0, m, b0, b1, A, B,df = on_submit(0)
# Create checkboxes for each column
column_checkboxes = [widgets.Checkbox(value=False, description=col) for col in df.columns]
column_box = widgets.VBox(column_checkboxes)

# Create checkboxes for export formats
export_options = ['CSV', 'Histogram', 'Comma-separated list']
export_checkboxes = [widgets.Checkbox(value=False, description=opt) for opt in export_options]
export_box = widgets.VBox(export_checkboxes)

# Bin slider for histogram
bin_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=500,
    step=5,
    description='Bins:',
    style={'description_width': 'initial'}
)

# Export button
export_button = widgets.Button(description='Export', button_style='success')

# Output area
output_area = widgets.Output()

# Callback function for export
def export_callback(b):
    with output_area:
        clear_output()
        selected_cols = [cb.description for cb in column_checkboxes if cb.value]
        selected_exports = [cb.description for cb in export_checkboxes if cb.value]
        bins = bin_slider.value

        if not selected_cols:
            print("⚠️ Please select at least one column.")
            return
        if not selected_exports:
            print("⚠️ Please select at least one export format.")
            return

        selected_df = df[selected_cols]

        if 'CSV' in selected_exports:
            filename = 'exported_data.csv'
            selected_df.to_csv(filename, index=False)
            display(FileLink(filename))
            print(f"✅ CSV exported to '{filename}'")

        if 'Histogram' in selected_exports:
            os.makedirs("histograms", exist_ok=True)
            for col in selected_cols:
                data = df[col].dropna()
                counts, edges = np.histogram(data, bins=bins)

                # Save histogram image
                plt.figure()
                plt.hist(data, bins=bins)
                plt.title(f'Histogram of {col}')
                plt.xlabel(col)
                plt.ylabel('Frequency')
                plt.grid(True)
                img_path = f"histograms/{col}_histogram.png"
                plt.savefig(img_path)
                plt.close()

                # Save histogram data as CSV
                hist_df = pd.DataFrame({
                    'Bin Start': edges[:-1],
                    'Bin End': edges[1:],
                    'Count': counts
                })
                csv_path = f"histograms/{col}_histogram_data.csv"
                hist_df.to_csv(csv_path, index=False)
                

                display(FileLink(csv_path))
                display(FileLink(img_path))

                print(f"📊 Histogram for '{col}' saved as image and CSV in 'histograms/'")

        if 'Comma-separated list' in selected_exports:
            for col in selected_cols:
                values = ', '.join(map(str, df[col].tolist()))
                print(f"{col}: {values}")



def query_callback(b):
    with query_output:
        clear_output()
        selected_cols = [cb.description for cb in column_checkboxes if cb.value]
        if not selected_cols:
            print("⚠️ Please select at least one column.")
            return

        custom_percentile = percentile_input.value
        include_ci = ci_checkbox.value  # ✅ Checkbox value

        print(f"📊 Querying statistics for selected columns with custom percentile: {custom_percentile}%\n")

        for col in selected_cols:
            data = df[col].dropna()
            n = len(data)
            if not np.issubdtype(data.dtype, np.number):
                print(f"❌ Column '{col}' is not numeric. Skipping.\n")
                continue

            mean = data.mean()
            median = data.median()
            std = data.std()
            var = data.var()
            data_range = data.max() - data.min()
            mode = data.mode().tolist()
            q25 = np.percentile(data, 25)
            q75 = np.percentile(data, 75)
            custom_q = np.percentile(data, custom_percentile)

            print(f"📈 Column: {col}")
            print(f"  Count (n): {n}")
            print(f"  Mean: {mean:.2f}")
            print(f"  Median: {median:.2f}")
            print(f"  Standard Deviation: {std:.2f}")
            print(f"  Variance: {var:.2f}")
            print(f"  Range (max - min): {data_range:.2f}")
            print(f"  Mode: {mode}")
            print(f"  25th Percentile: {q25:.2f}")
            print(f"  75th Percentile: {q75:.2f}")
            print(f"  {custom_percentile}th Percentile: {custom_q:.2f}")

            # ✅ Confidence Interval section
            if include_ci:
                if n < 10:
                    print(f"  ⚠️ Too few samples ({n}) for a reliable confidence interval.")
                else:
                    sem = stats.sem(data)
                    ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=sem)
                    print(f"  95% Confidence Interval for Mean: [{ci[0]:.2f}, {ci[1]:.2f}]")

            # 🧭 “What You Should Know” diagnostic section
            print("\n  🧭 What You Should Know:")
            if n < 10:
                print("   • ⚠️ Very small sample — interpret all stats with extreme caution.")
            elif n < 30:
                print("   • ⚠️ Small sample — estimates may vary widely if data changes.")
            elif n < 100:
                print("   • ✅ Moderate reliability — results are fairly stable.")
            else:
                print("   • 🟢 Large dataset — results are statistically robust.")

            print("\n" + "-"*60 + "\n")

# Custom percentile input
percentile_input = widgets.BoundedFloatText(
    value=50.0,
    min=0.0,
    max=100.0,
    step=0.1,
    description='Custom Percentile:',
    style={'description_width': 'initial'}
)

ci_checkbox = widgets.Checkbox(
    value=False,
    description='Include 95% Confidence Intervals',
    style={'description_width': 'initial'}
)

twoopts=widgets.HBox([percentile_input,ci_checkbox])

export_button.on_click(export_callback)

# Export tab
exp_box = widgets.VBox([
    widgets.Label("✅ Select columns to export:"),
    column_box,
    widgets.Label("📦 Choose export formats:"),
    export_box,
    bin_slider,
    export_button,
    output_area
])

# Query button
query_button = widgets.Button(description='Query Selected Columns', button_style='info')

# Output area for query results
query_output = widgets.Output()

query_button.on_click(query_callback)

# Query tab
query_box = widgets.VBox([
    widgets.Label("🔍 Select columns to query:"),
    column_box,
    twoopts,
    query_button,
    query_output
])

# Tabs
tab2_nest = widgets.Tab()
tab2_nest.children = [query_box, exp_box]
tab2_nest.set_title(0, 'Statistical Query')
tab2_nest.set_title(1, 'Export Data')
tab2_nest
